In [31]:
import open3d as o3d
import numpy as np
import os
import lib3mf
from lib3mf_common import *

model_name = "pi 3mm side paint"
model_path = f'input_models/{model_name}.3mf'
stl_path = f'input_models/{model_name}.stl'

if os.path.isfile(model_path):
    print("Found file!")
else:
    print("Did not find file!")
wrapper = get_wrapper()
model = wrapper.CreateModel()
read_3mf_file_to_model(model, model_path)

# convert to STL
writer = model.QueryWriter("stl")
print(f"Writing {stl_path}...")
writer.WriteToFile(stl_path)
print("Done")

print("Testing mesh in Open3D...")
mesh = o3d.io.read_triangle_mesh(stl_path)
print(mesh)
print('Vertices:')
print(np.asarray(mesh.vertices))
print('Triangles:')
print(np.asarray(mesh.triangles))

Found file!
Writing input_models/pi 3mm side paint.stl...
Done
Testing mesh in Open3D...
TriangleMesh with 619 points and 870 triangles.
Vertices:
[[ 25.          -5.          10.        ]
 [ 22.5         -5.          10.        ]
 [ 25.          -2.5         10.        ]
 ...
 [-13.23529053   5.          14.28571415]
 [-16.1764679    5.          14.28571415]
 [-19.11764526   5.          14.28571415]]
Triangles:
[[  0   1   2]
 [  2   1   3]
 [  2   3   4]
 ...
 [570 617 618]
 [566 570 618]
 [566 618 564]]


In [32]:
print("Try to render a mesh with normals (exist: " +
      str(mesh.has_vertex_normals()) + ") and colors (exist: " +
      str(mesh.has_vertex_colors()) + ")")
o3d.visualization.draw_geometries([mesh])
print("A mesh with no normals and no colors does not look good.")

Try to render a mesh with normals (exist: True) and colors (exist: False)
[Open3D WARNING] GLFW initialized for headless rendering.
[Open3D WARNING] GLFW Error: OSMesa: Library not found
[Open3D WARNING] Failed to create window
[Open3D WARNING] [DrawGeometries] Failed creating OpenGL window.
A mesh with no normals and no colors does not look good.


In [78]:
import networkx as nx
import numpy as np
import plotly.graph_objects
import pyvista as pv
import tetgen
from scipy.optimize import minimize, least_squares
from scipy.spatial.transform import Rotation as R
import open3d as o3d
import time
import pickle
import base64
import stl
from stl import mesh
import os
import lib3mf
from lib3mf_common import *
import copy

def process_mesh(model_mesh):
    def encode_object(obj):
        return base64.b64encode(pickle.dumps(obj)).decode('utf-8')
    
    up_vector = np.array([0, 0, 1])

    def decode_object(encoded_str):
        return pickle.loads(base64.b64decode(encoded_str))

    # process mesh as in S4_Slicer and return extracted surface

    input_tet = tetgen.TetGen(np.asarray(model_mesh.vertices), np.asarray(model_mesh.triangles))
    input_tet.tetrahedralize()
    input_tet = input_tet.grid

    PART_OFFSET = np.array([0., 0., 0.])
    x_min, x_max, y_min, y_max, z_min, z_max = input_tet.bounds
    input_tet.points -= np.array([(x_min + x_max) / 2, (y_min + y_max) / 2, z_min]) + PART_OFFSET


    # find neighbours
    cell_neighbour_dict = {neighbour_type: {face: [] for face in range(input_tet.number_of_cells)} for neighbour_type in ["point", "edge", "face"]}
    for neighbour_type in ["point", "edge", "face"]:
        cell_neighbours = []
        for cell_index in range(input_tet.number_of_cells):
            neighbours = input_tet.cell_neighbors(cell_index, f"{neighbour_type}s")
            for neighbour in neighbours:
                if neighbour > cell_index:
                    cell_neighbours.append((cell_index, neighbour))
        for face_1, face_2 in np.array(cell_neighbours):
            cell_neighbour_dict[neighbour_type][face_1].append(face_2)
            cell_neighbour_dict[neighbour_type][face_2].append(face_1)

        input_tet.field_data[f"cell_{neighbour_type}_neighbours"] = np.array(cell_neighbours)

    cell_neighbour_graph = nx.Graph()
    cell_centers = input_tet.cell_centers().points
    for edge in input_tet.field_data["cell_point_neighbours"]: # use point neighbours for best accuracy
        distance = np.linalg.norm(cell_centers[edge[0]] - cell_centers[edge[1]])
        cell_neighbour_graph.add_weighted_edges_from([(edge[0], edge[1], distance)])

    def update_tet_attributes(tet):
        '''
        Calculate face normals, face centers, cell centers, and overhang angles for each cell in the tetrahedral mesh.
        '''

        surface_mesh = tet.extract_surface()
        cell_to_face = decode_object(tet.field_data["cell_to_face"])

        # put general data in field_data for easy access
        cells = tet.cells.reshape(-1, 5)[:, 1:] # assume all cells have 4 vertices
        tet.add_field_data(cells, "cells")
        cell_vertices = tet.points
        tet.add_field_data(cell_vertices, "cell_vertices")
        faces = surface_mesh.faces.reshape(-1, 4)[:, 1:] # assume all faces have 3 vertices
        tet.add_field_data(faces, "faces")
        face_vertices = surface_mesh.points
        tet.add_field_data(face_vertices, "face_vertices")

        tet.cell_data['face_normal'] = np.full((tet.number_of_cells, 3), np.nan)
        surface_mesh_face_normals = surface_mesh.face_normals
        for cell_index, face_indices in cell_to_face.items():
            face_normals = surface_mesh_face_normals[face_indices]
            # get the normal facing the most down
            most_down_normal_index = np.argmin(face_normals[:, 2])
            tet.cell_data['face_normal'][cell_index] = face_normals[most_down_normal_index]
        tet.cell_data['face_normal'] =  tet.cell_data['face_normal'] / np.linalg.norm(tet.cell_data['face_normal'], axis=1)[:, None]

        tet.cell_data['face_center'] = np.empty((tet.number_of_cells, 3))
        tet.cell_data['face_center'][:,:] = np.nan
        surface_mesh_cell_centers = surface_mesh.cell_centers().points
        for cell_index, face_indices in cell_to_face.items():
            face_centers = surface_mesh_cell_centers[face_indices]
            # get the normal facing the most down
            most_down_center_index = np.argmin(face_centers[:, 2])
            tet.cell_data['face_center'][cell_index] = face_centers[most_down_center_index]

        tet.cell_data["cell_center"] = tet.cell_centers().points

        # calculate bottom cells
        bottom_cell_threshold = np.nanmin(tet.cell_data['face_center'][:, 2])+0.3
        bottom_cells_mask = tet.cell_data['face_center'][:, 2] < bottom_cell_threshold
        tet.cell_data['is_bottom'] = bottom_cells_mask
        bottom_cells = np.where(bottom_cells_mask)[0]

        face_normals = tet.cell_data['face_normal'].copy()
        face_normals[bottom_cells_mask] = np.nan # make bottom faces not angled
        overhang_angle = np.arccos(np.dot(face_normals, up_vector))
        tet.cell_data['overhang_angle'] = overhang_angle

        overhang_direction = face_normals[:, :2].copy()
        overhang_direction /= np.linalg.norm(overhang_direction, axis=1)[:, None]
        tet.cell_data['overhang_direction'] = overhang_direction

        # calculate if cell will print in air by seeing if any cell centers along path to base are higher
        IN_AIR_THRESHOLD = 1
        tet.cell_data['in_air'] = np.full(tet.number_of_cells, False)

        _, paths_to_bottom = nx.multi_source_dijkstra(cell_neighbour_graph, set(bottom_cells))

        # put it in cell data
        tet.cell_data['path_to_bottom'] = np.full((tet.number_of_cells, np.max([len(x) for x in paths_to_bottom.values()])), -1)
        for cell_index, path_to_bottom in paths_to_bottom.items():
            tet.cell_data['path_to_bottom'][cell_index, :len(path_to_bottom)] = path_to_bottom

        # calculate if cell is in air
        for cell_index in range(tet.number_of_cells):
            path_to_bottom = paths_to_bottom[cell_index]
            if len(path_to_bottom) > 1:
                cell_heights = tet.cell_data['cell_center'][path_to_bottom, 2]
                if np.any(cell_heights > tet.cell_data['cell_center'][cell_index, 2] + IN_AIR_THRESHOLD):
                    tet.cell_data['in_air'][cell_index] = True

        return tet

    def calculate_tet_attributes(tet):
        '''
        Calculate shared vertices between cells, cell to face & face to cell relations, and bottom cells of the tetrahedral mesh.
        '''

        surface_mesh = tet.extract_surface()

        # put general data in field_data for easy access
        cells = tet.cells.reshape(-1, 5)[:, 1:] # assume all cells have 4 vertices
        tet.add_field_data(cells, "cells")
        cell_vertices = tet.points
        tet.add_field_data(cell_vertices, "cell_vertices")
        faces = surface_mesh.faces.reshape(-1, 4)[:, 1:] # assume all faces have 3 vertices
        tet.add_field_data(faces, "faces")
        face_vertices = surface_mesh.points
        tet.add_field_data(face_vertices, "face_vertices")

        # calculate shared vertices
        shared_vertices = []
        for cell_1, cell_2 in tet.field_data["cell_point_neighbours"]:
            shared_vertices_these_faces = np.intersect1d(cells[cell_1], cells[cell_2])
            for vertex in shared_vertices_these_faces:
                shared_vertices.append({
                        "cell_1_index": cell_1,
                        "cell_2_index": cell_2,
                        "cell_1_vertex_index": np.where(cells[cell_1] == vertex)[0][0],
                        "cell_2_vertex_index": np.where(cells[cell_2] == vertex)[0][0],
                    })

        # calculate cell to face & face to cell relations
        cell_to_face = {}
        face_to_cell = {face_index: [] for face_index in range(len(faces))}
        cell_to_face_vertices = {}
        face_to_cell_vertices = {}
        for cell_vertex_index, cell_vertex in enumerate(tet.field_data["cell_vertices"].reshape(-1, 3)):
            face_vertex_index = np.where((face_vertices == cell_vertex).all(axis=1))[0]
            if len(face_vertex_index) == 1:
                cell_to_face_vertices[cell_vertex_index] = face_vertex_index[0]
                face_to_cell_vertices[face_vertex_index[0]] = cell_vertex_index

        for cell_index, cell in enumerate(tet.field_data["cells"]):
            face_vertex_indices = [cell_to_face_vertices[cell_vertex_index] for cell_vertex_index in cell if cell_vertex_index in cell_to_face_vertices]
            if len(face_vertex_indices) >= 3:
                extracted = surface_mesh.extract_points(face_vertex_indices, adjacent_cells=False)
                if extracted.number_of_cells >= 1:
                    cell_to_face[cell_index] = list(extracted.cell_data['vtkOriginalCellIds'])
                    for face_index in extracted.cell_data['vtkOriginalCellIds']:
                        face_to_cell[face_index].append(cell_index)

        tet.add_field_data(encode_object(cell_to_face), "cell_to_face")
        tet.add_field_data(encode_object(face_to_cell), "face_to_cell")

        # calculate has_face attribute
        tet.cell_data['has_face'] = np.zeros(tet.number_of_cells)
        for cell_index, face_indices in cell_to_face.items():
            tet.cell_data['has_face'][cell_index] = 1

        tet = update_tet_attributes(tet)

        # calculate bottom cells
        bottom_cells_mask = tet.cell_data['is_bottom']
        bottom_cells = np.where(bottom_cells_mask)[0]

        tet.cell_data['overhang_angle'][bottom_cells] = np.nan

        return tet, bottom_cells_mask, bottom_cells


    bottom_cells_mask = None
    bottom_cells = None
    input_tet, bottom_cells_mask, bottom_cells = calculate_tet_attributes(input_tet)

    # find bottom cell groups that are connected
    bottom_cell_graph = nx.Graph()
    for cell_index in bottom_cells:
        bottom_cell_graph.add_node(cell_index)
    cell_point_neighbour_dict = cell_neighbour_dict["point"]
    for cell_index in bottom_cells:
        for neighbour in cell_point_neighbour_dict[cell_index]:
            if neighbour in bottom_cells:
                bottom_cell_graph.add_edge(cell_index, neighbour)

    bottom_cell_groups = [list(x) for x in list(nx.connected_components(bottom_cell_graph))]

    undeformed_tet = input_tet.copy()

    surface = input_tet.extract_surface() #pyvista
    return surface

# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html with edits
def tri_to_processed_pt_cld(tri_mesh, voxel_size, num_points):
    pcd = tri_mesh.sample_points_uniformly(num_points)

    print(":: Downsample with a voxel size %.3f." % voxel_size)
    pcd_down = pcd.voxel_down_sample(voxel_size)

    radius_normal = voxel_size * 2
    print(":: Estimate normal with search radius %.3f." % radius_normal)
    pcd_down.estimate_normals(
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_normal, max_nn=30))

    radius_feature = voxel_size * 5
    print(":: Compute FPFH feature with search radius %.3f." % radius_feature)
    pcd_fpfh = o3d.pipelines.registration.compute_fpfh_feature(
        pcd_down,
        o3d.geometry.KDTreeSearchParamHybrid(radius=radius_feature, max_nn=100))
    return pcd_down, pcd_fpfh
    
# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html
def execute_global_registration(source_down, target_down, source_fpfh,
                                target_fpfh, voxel_size):
    distance_threshold = voxel_size * 1.5
    print(":: RANSAC registration on downsampled point clouds.")
    print("   Since the downsampling voxel size is %.3f," % voxel_size)
    print("   we use a liberal distance threshold %.3f." % distance_threshold)
    result = o3d.pipelines.registration.registration_ransac_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh, True,
        distance_threshold,
        o3d.pipelines.registration.TransformationEstimationPointToPoint(False),
        3, [
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnEdgeLength(
                0.9),
            o3d.pipelines.registration.CorrespondenceCheckerBasedOnDistance(
                distance_threshold)
        ], o3d.pipelines.registration.RANSACConvergenceCriteria(100000, 0.999))
    return result

# from https://www.open3d.org/docs/0.12.0/tutorial/pipelines/global_registration.html
def execute_fast_global_registration(source_down, target_down, source_fpfh,
                                     target_fpfh, voxel_size):
    distance_threshold = voxel_size * 0.5
    print(":: Apply fast global registration with distance threshold %.3f" \
            % distance_threshold)
    result = o3d.pipelines.registration.registration_fgr_based_on_feature_matching(
        source_down, target_down, source_fpfh, target_fpfh,
        o3d.pipelines.registration.FastGlobalRegistrationOption(
            maximum_correspondence_distance=distance_threshold))
    return result

def local_registration():
    return

import plotly.graph_objects as go
def draw_point_clouds(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.transform(transformation)
    source_temp = np.asarray(source_temp.points)
    target_temp = np.asarray(target_temp.points)
    # use plotly for display point clouds; open3d inline display with jupyter notebook is finicky
    fig = go.Figure(data=[go.Scatter3d(
        x = source_temp[:, 0], y = source_temp[:, 1], z = source_temp[:, 2],
        mode="markers", marker=dict(color="blue", size=1)),
        go.Scatter3d(
        x = target_temp[:, 0], y = target_temp[:, 1], z = target_temp[:, 2],
        mode="markers", marker=dict(color="orange", size=1))]
    )
    fig.update_layout(scene=dict(aspectmode='data')) # sets all axes to be equal so model is not distorted
    fig.show()

# from https://www.open3d.org/docs/release/tutorial/pipelines/global_registration.html
def draw_registration_result(source, target, transformation):
    source_temp = copy.deepcopy(source)
    target_temp = copy.deepcopy(target)
    source_temp.paint_uniform_color([1, 0.706, 0])
    target_temp.paint_uniform_color([0, 0.651, 0.929])
    source_temp.transform(transformation)
    o3d.visualization.draw_geometries([source_temp, target_temp],
                                      zoom=0.4559,
                                      front=[0.6452, -0.3036, -0.7011],
                                      lookat=[1.9892, 2.0208, 1.8945],
                                      up=[-0.2779, -0.9482, 0.1556])

# expects to be run from S4_Slicer folder
# Load 3MF
model_name = "pi 3mm side paint"
model_path = f'input_models/{model_name}.3mf'
stl_path = f'input_models/{model_name}.stl'

if os.path.isfile(model_path):
    print("Found file!")
else:
    print("Did not find file!")
    sys.exit()
wrapper = get_wrapper()
model = wrapper.CreateModel()
read_3mf_file_to_model(model, model_path)

# convert to STL
writer = model.QueryWriter("stl")
print(f"Writing {stl_path}...")
writer.WriteToFile(stl_path)
print("Done")

mf_mesh = o3d.io.read_triangle_mesh(stl_path)
# get o3d triangle mesh from pyvista PolyData containing vertices, triangles
# we do this to allow downsampling in case we use large meshes
disordered_mesh = process_mesh(mf_mesh)
o3d_vert = o3d.utility.Vector3dVector(disordered_mesh.points)
o3d_tri = o3d.utility.Vector3iVector(disordered_mesh.faces.reshape(-1, 4)[:, 1:])
disordered_mesh = o3d.geometry.TriangleMesh(o3d_vert, o3d_tri)

# draw to check scene
# draw_registration_result(mf_mesh, disordered_mesh, np.eye(4))

# check disordered mesh vs 3mf vertex list
mf_mesh = model.GetMeshObjectByID(1)
mf_verts = []
for i in range(mf_mesh.GetVertexCount()):
    vert = [x for x in mf_mesh.GetVertex(i).Coordinates]
    mf_verts.append(vert)
mf_verts = np.asarray(mf_verts)

mf_tri = []
for i in range(mf_mesh.GetTriangleCount()):
    tri = [y for y in mf_mesh.GetTriangle(i).Indices]
    mf_tri.append(tri)
mf_tri = np.asarray(mf_tri)
mf_verts = o3d.utility.Vector3dVector(mf_verts)
mf_tri = o3d.utility.Vector3iVector(mf_tri)
mf_mesh = o3d.geometry.TriangleMesh(mf_verts, mf_tri)
# draw to check scene
# draw_registration_result(mf_mesh, disordered_mesh, np.eye(4))

# generate downsampled point cloud on both for RANSAC
voxel_size = 0.05
num_pts = 2500
mf_cld, mf_fpfh = tri_to_processed_pt_cld(mf_mesh, voxel_size, num_pts)
disordered_cld, disordered_fpfh = tri_to_processed_pt_cld(disordered_mesh,voxel_size, num_pts)
draw_point_clouds(mf_cld, disordered_cld, np.eye(4))

# run RANSAC for global registration
result_ransac = execute_fast_global_registration(source_down=mf_cld, target_down=disordered_cld,
                                            source_fpfh=mf_fpfh, target_fpfh=disordered_fpfh,
                                            voxel_size=voxel_size)
print(result_ransac)
print("RANSAC transformation is:\n", result_ransac.transformation)
draw_point_clouds(mf_cld, disordered_cld, result_ransac.transformation)

# run ICP for local registration - we need tight alignment of meshes so correspondence works
# essentially, alignment tolerance is the tolerance of np.isclose()
threshold = 1
trans_init = result_ransac.transformation
print("Apply point-to-point ICP")
reg_p2p = o3d.pipelines.registration.registration_icp(
    mf_cld, disordered_cld, threshold, trans_init,
    o3d.pipelines.registration.TransformationEstimationPointToPoint(),
    o3d.pipelines.registration.ICPConvergenceCriteria(max_iteration=2000))
print(reg_p2p)
print("Local Transformation is:")
print(reg_p2p.transformation)
draw_point_clouds(mf_cld, disordered_cld, reg_p2p.transformation)

correspondence_transfo = reg_p2p.transformation

Found file!
Writing input_models/pi 3mm side paint.stl...
Done
:: Downsample with a voxel size 0.050.
:: Estimate normal with search radius 0.100.
:: Compute FPFH feature with search radius 0.250.
:: Downsample with a voxel size 0.050.
:: Estimate normal with search radius 0.100.
:: Compute FPFH feature with search radius 0.250.


/tmp/ipykernel_55638/2248002612.py:110: RuntimeWarning:

invalid value encountered in divide



:: Apply fast global registration with distance threshold 0.025
RegistrationResult with fitness=4.006410e-04, inlier_rmse=2.253390e-02, and correspondence_set size of 1
Access transformation to get result.
RANSAC transformation is:
 [[ 1.         -0.          0.          0.33877247]
 [-0.          1.         -0.          0.0113778 ]
 [-0.         -0.          1.         10.1996101 ]
 [-0.          0.         -0.          1.        ]]


Apply point-to-point ICP
RegistrationResult with fitness=9.403045e-01, inlier_rmse=5.512400e-01, and correspondence_set size of 2347
Access transformation to get result.
Local Transformation is:
[[ 9.99996538e-01  8.07900842e-04  2.50417841e-03  3.00228328e-02]
 [-8.14931095e-04  9.99995726e-01  2.80766130e-03 -4.29855423e-03]
 [-2.50189939e-03 -2.80969231e-03  9.99992923e-01  9.98894569e+00]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


In [89]:
# use correspondence_transformation to map correspondence
import lib3mf
from lib3mf_common import *

def list_3mf_verts(model):
    verts = []
    for i in range(mf_num_verts):
        vert = [x for x in wrapper.GetMeshObjectByID(1).GetVertex(i).Coordinates]
        verts.append(vert)
    return np.asarray(verts)

# TESTING VERSION -- just apply the shift from RANSAC
# this tells us if RANSAC is accurate enough or if we need local registratoin
def shift_vert_to_center(lib3mf_Vertex):
    x, y, z = lib3mf_Vertex.Coordinates
    x = x - 0 # for pi 3mm painted? Why? Yes this and side painted
    y = y - 0
    z = z + 10
    # # build rigidbody transformation matrix
    # # 3mf verts do not include rotation so identity
    # V = np.eye(4)
    # V[:3, 3] = [x, y, z]
    # Vn = V @ correspondence_transfo
    # x = Vn[0][3]
    # y = Vn[1][3]
    # z = Vn[2][3]
    new_vert = lib3mf.Position()
    new_vert.Coordinates[0] = x
    new_vert.Coordinates[1] = y
    new_vert.Coordinates[2] = z
    return new_vert

# this is a little stupid but you DO have to double wrap it
def mf_vert_to_dbl_np_array(lib3mf_Vertex):
    x, y, z = lib3mf_Vertex.Coordinates
    return np.asarray([[x, y, z]]) 

# load 3mf
# getting vertices from lib3mf
if os.path.isfile(model_path):
    wrapper = get_wrapper()
    wrapper = wrapper.CreateModel()
    read_3mf_file_to_model(wrapper, model_path)
    mesh_obj = wrapper.GetMeshObjectByID(1) # NOTE: even for multimaterial models all vertices are saved in model id 1.
    # doing a deformation over all vertices (subtract bounding box)
    mf_num_verts = mesh_obj.GetVertexCount()
    for i in range(mf_num_verts):
        new_vect = shift_vert_to_center(mesh_obj.GetVertex(i))
        mesh_obj.SetVertex(i, new_vect)

surf_verts = np.asarray(disordered_mesh.vertices)

mf_num_verts = mesh_obj.GetVertexCount()

sample = 10
print(f"First {sample} surface verts:")
print(surf_verts[:sample])
print(f"First {sample} 3mf verts:")
mf_verts = list_3mf_verts(model)
print(mf_verts[:sample])
print(f"First {sample} mf verts - surf verts")
print(mf_verts[:sample] - surf_verts[:sample])

check_verts = surf_verts
check_range = mf_num_verts
correspondence_list = np.full(check_range, -1)
for i in range(check_range):
    curr_vert = mesh_obj.GetVertex(i)
    curr_vert = mf_vert_to_dbl_np_array(curr_vert)
    match_vert_mask = (np.isclose(check_verts[:, :],curr_vert[:, None])).all(axis=-1).any(axis=0) 
    match_indices = np.where(match_vert_mask)

    if len(match_indices[0]) > 0:
        correspondence_list[i] = (match_indices[0][0])

if -1 in correspondence_list:
    print("No correspondence")
else:
    print("YAY found correspondence")
print(correspondence_list)

First 10 surface verts:
[[22.5 -2.5 10. ]
 [20.   0.  10. ]
 [22.5  0.  10. ]
 [25.  -5.  10. ]
 [22.5 -5.  10. ]
 [25.   0.  10. ]
 [20.  -5.  10. ]
 [20.  -2.5 10. ]
 [25.  -2.5 10. ]
 [25.   2.5 12.5]]
First 10 3mf verts:
[[25.  -5.  10. ]
 [22.5 -5.  10. ]
 [25.  -2.5 10. ]
 [22.5 -2.5 10. ]
 [25.   0.  10. ]
 [22.5  0.  10. ]
 [25.   2.5 10. ]
 [22.5  2.5 10. ]
 [25.   5.  10. ]
 [22.5  5.  10. ]]
First 10 mf verts - surf verts
[[ 2.5 -2.5  0. ]
 [ 2.5 -5.   0. ]
 [ 2.5 -2.5  0. ]
 [-2.5  2.5  0. ]
 [ 2.5  5.   0. ]
 [-2.5  0.   0. ]
 [ 5.   7.5  0. ]
 [ 2.5  5.   0. ]
 [ 0.   7.5  0. ]
 [-2.5  2.5 -2.5]]
YAY found correspondence
[  3   4   8   0   5   2  13  12  16  17  14  18  20  21  25  23   6   7
   1  28  29  22  33  34  24  39  38  41  42  46  48  50  49  54  53  51
  52   9  15  45  44  47  57  11  10  66  59  64  61  69  68  75  74  77
  79  76  78  81  82  88  87  93  91  96  97 101  99 104 105 107 108 113
 112 116 115 117 118 124 123 127 128 132 131 136 135  67  60  72 

In [91]:
# With tight alignment, build a KDTree to find nearest neighbors
# this is necessary because even tuned ICP does not provide a tight enough transformation
# to return np.isclose() true
# any nearest-neighbors algorithm would work

from scipy.spatial import KDTree
# throw surf vert into KDtree, then can use mf_vert as queries to tree
tree = KDTree(surf_verts)

# query for nearest neighbor of first mf_vert; if same as manual offsets, should return 3 in index
correspondence_list = np.full(check_range, -1)
for i in range(mf_num_verts):
    v = mf_verts[i]
    d, j = tree.query(v)
    correspondence_list[i] = j

if -1 in correspondence_list:
    print("No correspondence")
else:
    print("YAY found correspondence")
print(correspondence_list)

YAY found correspondence
[  3   4   8   0   5   2  13  12  16  17  14  18  20  21  25  23   6   7
   1  28  29  22  33  34  24  39  38  41  42  46  48  50  49  54  53  51
  52   9  15  45  44  47  57  11  10  66  59  64  61  69  68  75  74  77
  79  76  78  81  82  88  87  93  91  96  97 101  99 104 105 107 108 113
 112 116 115 117 118 124 123 127 128 132 131 136 135  67  60  72 138 139
  83 142 143  89 146 144  92 147 148  95 151 152 100 153 154 103 156 155
 109 158 157 111 159 160 114 166 165 119 168 169 125 171 172 126 173 174
 133  40 176 134 177 178 181 179 186 188 190 189 194 196 191 197  73  80
 184 185 187  62  71  70 204 200 203 207 209 212 214 216 218 217 215 221
 193 192 205 206 213 183 182 195 227 228 224 225 232 226 237 236 241 242
 239 243 220 219 233 234 235 208 211 210 255 248 250 249 259 258 261 262
 265 264 263 266 238 240 253 254 260 230 229 231 273 276 271 277 278 279
 283 285 289 287 286 290 268 267 280 281 282 251 256 257 305 302 301 297
 300 299 307 306 314 313 3

In [ ]:
# TEST BLOCK BROKEN
# cannot find transformation in build items
print(model.GetBuildItems())
print(model.GetBuildItems().Count())
iter = model.GetBuildItems()
# print(iter.GetCurrent())
# print(model.GetBuildItems().GetCurrent().GetObjectTransform())

# can find in components?
print("Components")
print(model.GetComponentsObjects())
# print(model.GetComponentsObjects().GetCurrentComponentsObject())
# print(model.GetComponentsObjectByID(1))
# print(model.GetComponentsObjectByID(1).GetComponent(1))